# Heart Disease Prediction System

*Academic machine-learning project (educational use only, **not** a medical diagnosis).*

Modified from the instructor's **Titanic Passenger Survival Prediction** notebook, following the same steps.

## Introduction
A binary classification task: predict whether a patient has **heart disease (1)** or **no heart disease (0)** from 4 simple attributes.

## Input and Output

| **Input Attribute** | **Meaning** | **Possible Values** |
|---|---|---|
| **sex** | Sex of patient | 0 = Female, 1 = Male |
| **cp** | Chest pain type | 1 = Typical angina, 2 = Atypical angina, 3 = Non-anginal pain, 4 = Asymptomatic |
| **thalach** | Maximum heart rate achieved (bpm) | Number (71 to 202 in the data) |
| **exang** | Exercise-induced chest pain | 0 = No, 1 = Yes |

| **Output Attribute** | **Possible Values** |
|---|---|
| **HeartDisease** | 0 = No Heart Disease, 1 = Heart Disease |

**Dataset:** UCI Heart Disease, Cleveland Clinic database (`processed.cleveland.data`, 303 patients).
The original target `num` has values 0-4. As the dataset documentation states, `0` = no disease and `1-4` = disease present, so we convert it to 0/1.

## Table of Contents
- Step 1: Import Libraries
- Step 2: Load Sample Data
- Step 3: Understand and Pre-process Sample Data
- Step 4: Feature Extraction (select exactly 4 features)
- Step 5: Label Encoding (not needed: all data is already numeric)
- Step 6: Execute the Training Phase
- Step 7: Execute the Testing Phase
- Step 8: Execute the Application Phase
- Step 9: Deploy as a Public Web App

### Step 1: Import Libraries

In [1]:
# numpy: numbers/arrays. pandas: tables (DataFrames). pickle: save/load the trained model.
import numpy as np
import pandas as pd
import pickle

# train_test_split: divides data into training data and testing data
from sklearn.model_selection import train_test_split

# svm: Support Vector Classifier (the same algorithm used in the Titanic example)
from sklearn import svm

# StandardScaler + make_pipeline: put numbers on a similar scale (explained in Step 6.3)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# accuracy_score: percentage of correct predictions
from sklearn.metrics import accuracy_score

### Step 2: Load Sample Data

In [2]:
# 'heart-data.csv' = the 4 selected input columns + the output column (from the Cleveland data)
sample_data = pd.read_csv("heart-data.csv")

print("Sample Data (first 10 rows):")
print("============================\n")
print(sample_data.head(10))

Sample Data (first 10 rows):

   sex  cp  thalach  exang  HeartDisease
0    1   1      150      0             0
1    1   4      108      1             1
2    1   4      129      1             1
3    1   3      187      0             0
4    0   2      172      0             0
5    1   2      178      0             0
6    0   4      160      0             1
7    0   4      163      1             0
8    1   4      147      0             1
9    1   4      155      1             1


### Step 3: Understand and Pre-process Sample Data
#### Step 3.1: Understand Sample Data

In [3]:
print("Attributes in Sample Data:")
print(sample_data.columns)

print("\nNumber of Instances in Sample Data:", sample_data["sex"].count())

print("\nMissing values per column:")
print(sample_data.isnull().sum())

print("\nOutput values (0 = No Heart Disease, 1 = Heart Disease):")
print(sample_data["HeartDisease"].value_counts())

Attributes in Sample Data:
Index(['sex', 'cp', 'thalach', 'exang', 'HeartDisease'], dtype='object')

Number of Instances in Sample Data: 303

Missing values per column:
sex             0
cp              0
thalach         0
exang           0
HeartDisease    0
dtype: int64

Output values (0 = No Heart Disease, 1 = Heart Disease):
HeartDisease
0    164
1    139
Name: count, dtype: int64


#### Step 3.2: Pre-process Sample Data
The original Cleveland file has 14 columns and the target `num` has values 0-4. We prepared `heart-data.csv` from it like this:
1. Kept only the 4 chosen inputs (`sex`, `cp`, `thalach`, `exang`).
2. Converted `num` into `HeartDisease`: `0` stays 0 (no disease), `1,2,3,4` become 1 (disease).

These 4 columns have **no missing values**, so no further cleaning is needed.

### Step 4: Feature Extraction
Exactly **4** input features are selected. Everything else in the original dataset is ignored.

In [4]:
feature_columns = ["sex", "cp", "thalach", "exang"]   # ORDER MATTERS: the web app must use this exact order
target_column = "HeartDisease"

print("Input features :", feature_columns)
print("Output (target):", target_column)

Input features : ['sex', 'cp', 'thalach', 'exang']
Output (target): HeartDisease


### Step 5: Label Encoding
**Not needed.** In the Titanic example the data was text (Male/Female, First/Second/Third), so it had to be converted to numbers. Our Heart Disease data is **already numeric**, so we do not use `LabelEncoder`.

### Step 6: Execute the Training Phase
#### Step 6.1: Splitting Sample Data into Training Data and Testing Data

In [5]:
# X = the 4 input features, y = the answer we want to predict
X = sample_data[feature_columns]
y = sample_data[target_column]

# 80% training data (to teach the model), 20% testing data (to check it on unseen patients)
# random_state=0  -> same split every time
# stratify=y      -> both parts keep the same share of disease / no-disease patients
input_vector_train, input_vector_test, output_label_train, output_label_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

print("Training data:", input_vector_train.shape, "| Testing data:", input_vector_test.shape)

Training data: (242, 4) | Testing data: (61, 4)


#### Step 6.2: Input Vectors and Output Labels
Already separated above: `input_vector_train` / `output_label_train` (training) and `input_vector_test` / `output_label_test` (testing).

#### Step 6.3: Train the Support Vector Classifier
Same algorithm as the Titanic example (SVC). One difference: Titanic's codes were all tiny (0 to 3). Here `thalach` is about 150 while the others are 0 to 4, so it would overpower them. `StandardScaler` puts every column on a similar scale. We combine the scaler and the SVC in one `pipeline`, so they are trained, saved and used together as a single model.

In [6]:
svc_model = make_pipeline(StandardScaler(), svm.SVC(gamma='auto', random_state=0))

# .fit() = learn from the training data
svc_model.fit(input_vector_train, np.ravel(output_label_train))
print(svc_model)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('svc', SVC(gamma='auto', random_state=0))])


#### Step 6.4: Save the Trained Model

In [7]:
pickle.dump(svc_model, open('heart_svc_model.pkl', 'wb'))
print("Saved: heart_svc_model.pkl")

Saved: heart_svc_model.pkl


### Step 7: Execute the Testing Phase

In [8]:
# Load the saved model (as the web app will do)
model = pickle.load(open('heart_svc_model.pkl', 'rb'))

# .predict() = model's answer for each test patient
model_predictions = model.predict(input_vector_test)

results = input_vector_test.copy()
results["Actual"] = output_label_test
results["Predicted"] = model_predictions
print(results.head(10))

     sex  cp  thalach  exang  Actual  Predicted
125    0   2      175      0       0          0
83     1   3      150      1       1          0
53     1   2      188      0       0          0
45     1   3      165      0       1          0
248    1   4      168      0       1          0
211    1   1      182      1       1          0
36     1   4      120      1       1          1
106    1   4      162      1       1          1
120    1   4      150      1       1          1
31     1   4      160      1       1          1


In [9]:
model_accuracy_score = accuracy_score(output_label_test, model_predictions)
print("Model Accuracy:", round(model_accuracy_score * 100, 2), "%")

Model Accuracy: 77.05 %


### Step 8: Execute the Application Phase

In [10]:
# Example patient (change these 4 values to try your own)
sex_input     = 1     # 0 = Female, 1 = Male
cp_input      = 4     # 1..4 chest pain type
thalach_input = 120   # maximum heart rate
exang_input   = 1     # 0 = No, 1 = Yes

# Same column order as training: sex, cp, thalach, exang
user_input = pd.DataFrame([[sex_input, cp_input, thalach_input, exang_input]], columns=feature_columns)
print(user_input)

model = pickle.load(open('heart_svc_model.pkl', 'rb'))
predicted = model.predict(user_input)[0]

if predicted == 1:
    print("\nPrediction: HEART DISEASE DETECTED")
else:
    print("\nPrediction: NO HEART DISEASE DETECTED")

   sex  cp  thalach  exang
0    1   4      120      1

Prediction: HEART DISEASE DETECTED


### Step 9: Deploy as a Public Web App
See `README.md`: push the project to GitHub, then deploy `app.py` with Streamlit Community Cloud to get a public URL.

> **Disclaimer:** educational machine learning project. It is not a medical diagnosis tool.